# Generate Single Stable Radar Dataset With Error Levels 1,2,3

This notebook generates a synthetic PRI dataset for one stable radar type.

The output dataset is saved under:
datasets/single_stable_1000x128_error_lvX/

## Imports

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml


# Error Level Definitions

In [2]:

ERROR_LEVELS = {
    "clean": {
        "poi": 1.0,
        "multipath_prob": 0.0,
        "aoa_drift_rate": 0.0,
        "toa_noise_std": 0.0,
        "freq_noise_std": 0.0,
        "pw_noise_std": 0.0,
        "amp_noise_std": 0.0,
        "aoa_noise_std": 0.0,
        "global_drop_rate": 0.0,
        "noise_rate": 0.0,
        "quantize": False,
    },

    # Very close to ideal
    "error_lv1": {
        "poi": 0.995,              # MPR = 0.5%
        "multipath_prob": 0.0,
        "aoa_drift_rate": 0.0,
        "toa_noise_std": 0.1,
        "freq_noise_std": 0.1,
        "pw_noise_std": 0.01,
        "amp_noise_std": 0.2,
        "aoa_noise_std": 0.05,
        "global_drop_rate": 0.0,
        "noise_rate": 1.0,
        "quantize": False,
    },

    # Mild but visible errors
    "error_lv2": {
        "poi": 0.99,               # MPR = 1%
        "multipath_prob": 0.01,
        "aoa_drift_rate": 0.2,
        "toa_noise_std": 0.5,
        "freq_noise_std": 0.5,
        "pw_noise_std": 0.03,
        "amp_noise_std": 0.5,
        "aoa_noise_std": 0.2,
        "global_drop_rate": 0.005,
        "noise_rate": 5.0,
        "quantize": False,
    },

    # Moderate nonideal case
    "error_lv3": {
        "poi": 0.98,               # MPR = 2%
        "multipath_prob": 0.03,
        "aoa_drift_rate": 0.5,
        "toa_noise_std": 1.0,
        "freq_noise_std": 1.0,
        "pw_noise_std": 0.05,
        "amp_noise_std": 1.0,
        "aoa_noise_std": 0.5,
        "global_drop_rate": 0.01,
        "noise_rate": 10.0,
        "quantize": False,
    },
}


# Paths

In [3]:

PROJECT_ROOT = Path("..").resolve()

RADAR_TYPE = "stable"
SEQUENCE_COUNT = 1000
PULSES_PER_SEQUENCE = 128
RANDOM_SEED = 42

GENERATOR_DIR = PROJECT_ROOT / "synthetic data generation"
DATASET_ROOT = PROJECT_ROOT / "datasets"


sys.path.append(str(GENERATOR_DIR))

from radar_generator import Environment, StableRadar

PROJECT_ROOT, GENERATOR_DIR

(WindowsPath('C:/Users/ciftc/OneDrive/Desktop/homeworks/4. year/1/EE 493/repo/Bitirme'),
 WindowsPath('C:/Users/ciftc/OneDrive/Desktop/homeworks/4. year/1/EE 493/repo/Bitirme/synthetic data generation'))

## Dataset Configuration

This dataset is intentionally simple.

Radar behavior:
- single stable radar
- fixed PRI
- fixed carrier frequency
- no missing pulses
- no false pulses
- no multipath
- no TOA noise
- no frequency noise
- no AOA drift
- stationary target
- no quantization

The purpose is to create a clean baseline dataset before adding more realistic effects.

In [4]:

RADAR_ID = "R_STABLE"
PRI_US = 1000.0
FREQ_MEAN_MHZ = 9000.0
PW_MEAN_US = 1.5
AMP_MEAN_DBM = -55.0
AOA_MEAN_DEG = 45.0

# Save CSV function definition

In [5]:
def save_individual_sequence_csvs(split_df, split_dir):
    for sequence_id, seq_df in split_df.groupby("sequence_id"):
        seq_df = seq_df.sort_values("pulse_index")
        seq_df.to_csv(split_dir / f"{sequence_id}.csv", index=False)

## Generate Pulse Sequences, Save Sequences.csv and Split Csv Files

Each `sequence_id` represents one independent radar pulse train.

The sequences should not be concatenated as one long continuous signal.
Models and algorithms should process each `sequence_id` separately.

In [6]:
for LEVEL, cfg in ERROR_LEVELS.items():
    DATASET_NAME = f"single_stable_1000x128_{LEVEL}"
    DATASET_DIR = DATASET_ROOT / DATASET_NAME

    TRAIN_DIR = DATASET_DIR / "train"
    VAL_DIR = DATASET_DIR / "val"
    TEST_DIR = DATASET_DIR / "test"

    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    VAL_DIR.mkdir(parents=True, exist_ok=True)
    TEST_DIR.mkdir(parents=True, exist_ok=True)

    all_sequences = []

    for i in range(SEQUENCE_COUNT):
        sequence_id = f"seq_{i:04d}"

        env = Environment()

        stable_radar = StableRadar(
            radar_id=RADAR_ID,
            pri=PRI_US,
            freq_mean=FREQ_MEAN_MHZ,
            freq_mode="constant",
            freq_hopping_values=None,
            poi=cfg["poi"],
            multipath_prob=cfg["multipath_prob"],
            multipath_delay_mean=1.5,
            multipath_delay_std=0.2,
            pw_mean=PW_MEAN_US,
            amp_mean=AMP_MEAN_DBM,
            aoa_mean=AOA_MEAN_DEG,
            scan_period=0.0,
            aoa_drift_rate=cfg["aoa_drift_rate"],
            stretch_pw_on_multipath=False,
        )

        env.add_radar(stable_radar)

        df_seq = env.generate(
            duration=PULSES_PER_SEQUENCE * PRI_US,
            toa_noise_std=cfg["toa_noise_std"],
            freq_noise_std=cfg["freq_noise_std"],
            pw_noise_std=cfg["pw_noise_std"],
            amp_noise_std=cfg["amp_noise_std"],
            aoa_noise_std=cfg["aoa_noise_std"],
            global_drop_rate=cfg["global_drop_rate"],
            noise_rate=cfg["noise_rate"],
            quantize=cfg["quantize"],
        )

        df_seq = df_seq.sort_values("TOA").reset_index(drop=True)

        df_seq["sequence_id"] = sequence_id
        df_seq["pulse_index"] = np.arange(len(df_seq))
        df_seq["radar_type"] = "stable"
        df_seq["error_level"] = LEVEL

        all_sequences.append(df_seq)

    df = pd.concat(all_sequences, ignore_index=True)

    # split by sequence_id
    sequence_ids = sorted(df["sequence_id"].unique())
    rng = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(sequence_ids)

    train_ids = set(sequence_ids[:700])
    val_ids = set(sequence_ids[700:800])
    test_ids = set(sequence_ids[800:1000])

    train_df = df[df["sequence_id"].isin(train_ids)].copy()
    val_df = df[df["sequence_id"].isin(val_ids)].copy()
    test_df = df[df["sequence_id"].isin(test_ids)].copy()

    train_df.to_csv(TRAIN_DIR / "sequences.csv", index=False)
    val_df.to_csv(VAL_DIR / "sequences.csv", index=False)
    test_df.to_csv(TEST_DIR / "sequences.csv", index=False)

    save_individual_sequence_csvs(train_df, TRAIN_DIR)
    save_individual_sequence_csvs(val_df, VAL_DIR)
    save_individual_sequence_csvs(test_df, TEST_DIR)

    # write metadata
    
    metadata = {
        "name": DATASET_NAME,
        "radar_type": RADAR_TYPE,
        "sequence_count": SEQUENCE_COUNT,
        "expected_pulses_per_sequence": PULSES_PER_SEQUENCE,
        "error_level": LEVEL,
        "error_parameters": cfg,
        "files": {
            "train": "train/sequences.csv",
            "val": "val/sequences.csv",
            "test": "test/sequences.csv",
            "individual_sequence_files": "Each split folder also contains one CSV per sequence, named seq_XXXX.csv.",
        },
        "split": {
            "train": {
                "ratio": 0.70,
                "sequence_count": train_df["sequence_id"].nunique(),
                "row_count": len(train_df),
            },
            "val": {
                "ratio": 0.10,
                "sequence_count": val_df["sequence_id"].nunique(),
                "row_count": len(val_df),
            },
            "test": {
                "ratio": 0.20,
                "sequence_count": test_df["sequence_id"].nunique(),
                "row_count": len(test_df),
            },
        },
        "radar_parameters": {
            "radar_id": RADAR_ID,
            "pri_us": PRI_US,
            "freq_mean_mhz": FREQ_MEAN_MHZ,
            "freq_mode": "constant",
            "poi": cfg["poi"],
            "multipath_prob": cfg["multipath_prob"],
            "aoa_drift_rate": cfg["aoa_drift_rate"],
            "pw_mean_us": PW_MEAN_US,
            "amp_mean_dbm": AMP_MEAN_DBM,
            "aoa_mean_deg": AOA_MEAN_DEG,
        },
        "environment_parameters": {
            "duration_us": PULSES_PER_SEQUENCE * PRI_US,
            "toa_noise_std": cfg["toa_noise_std"],
            "freq_noise_std": cfg["freq_noise_std"],
            "pw_noise_std": cfg["pw_noise_std"],
            "amp_noise_std": cfg["amp_noise_std"],
            "aoa_noise_std": cfg["aoa_noise_std"],
            "global_drop_rate": cfg["global_drop_rate"],
            "noise_rate": cfg["noise_rate"],
            "quantize": cfg["quantize"],
        },
        "units": {
            "TOA": "microseconds",
            "PRI": "microseconds",
            "Frequency": "MHz",
            "PW": "microseconds",
            "Amplitude": "dBm",
            "AOA": "degrees",
        },
    }

    with open(DATASET_DIR / "metadata.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(metadata, f, sort_keys=False)

    print(DATASET_NAME)
    print("Train:", train_df["sequence_id"].nunique(), "seqs,", len(train_df), "rows")
    print("Val:", val_df["sequence_id"].nunique(), "seqs,", len(val_df), "rows")
    print("Test:", test_df["sequence_id"].nunique(), "seqs,", len(test_df), "rows")


single_stable_1000x128_clean
Train: 700 seqs, 90300 rows
Val: 100 seqs, 12900 rows
Test: 200 seqs, 25800 rows
single_stable_1000x128_error_lv1
Train: 700 seqs, 89923 rows
Val: 100 seqs, 12847 rows
Test: 200 seqs, 25739 rows
single_stable_1000x128_error_lv2
Train: 700 seqs, 89754 rows
Val: 100 seqs, 12799 rows
Test: 200 seqs, 25627 rows
single_stable_1000x128_error_lv3
Train: 700 seqs, 89706 rows
Val: 100 seqs, 12797 rows
Test: 200 seqs, 25655 rows


In [7]:
readme = """# PRI Benchmark Datasets

This folder contains generated synthetic radar pulse datasets.

Each dataset is stored in its own subfolder. For example:

```text
datasets/
  single_stable_1000x128_clean/
  single_stable_1000x128_error_lv1/
  single_stable_1000x128_error_lv2/
  single_stable_1000x128_error_lv3/
```

Each dataset folder follows the same structure:

```text
dataset_name/
  metadata.yaml
  train/
    sequences.csv
    seq_XXXX.csv
  val/
    sequences.csv
    seq_XXXX.csv
  test/
    sequences.csv
    seq_XXXX.csv
```

## Dataset Naming

Dataset names follow this pattern:

```text
single_<radar_type>_<sequence_count>x<expected_pulses_per_sequence>_<error_level>
```

Example:

```text
single_stable_1000x128_error_lv1
```

This means:
- one radar type: stable
- 1000 independent sequences
- 128 expected transmitted pulses per sequence
- error level 1

## Error Levels

The error level describes how nonideal the generated radar observations are.

Current levels:

| Level | Meaning |
|---|---|
| `clean` | Ideal baseline with no missing pulses, no false pulses, and no measurement noise |
| `error_lv1` | Very close to ideal; tiny measurement noise and very small missing/false pulse effects |
| `error_lv2` | Mild nonideal case; visible but still moderate noise and pulse errors |
| `error_lv3` | Stronger nonideal case; more missing pulses, false pulses, multipath, and measurement noise |

The exact parameters for each dataset are saved in:

```text
metadata.yaml
```

Important fields in `metadata.yaml`:
- `error_level`
- `error_parameters`
- `radar_parameters`
- `environment_parameters`
- `split`

## Train, Validation, and Test Splits

Each dataset is split into:

```text
train/
val/
test/
```

For the 1000-sequence datasets:

| Split | Number of sequences |
|---|---:|
| train | 700 |
| val | 100 |
| test | 200 |

The split is done by `sequence_id`, not by individual rows. This prevents leakage between train, validation, and test data.

Leakage means the model accidentally sees part of the same sequence during training and testing, which makes evaluation unfair.

## Combined CSV Files

Each split contains a combined CSV file:

```text
train/sequences.csv
val/sequences.csv
test/sequences.csv
```

These files contain all sequences for that split.

Example:

```python
import pandas as pd

dataset_path = "datasets/single_stable_1000x128_error_lv1"

train_df = pd.read_csv(f"{dataset_path}/train/sequences.csv")
val_df = pd.read_csv(f"{dataset_path}/val/sequences.csv")
test_df = pd.read_csv(f"{dataset_path}/test/sequences.csv")
```

## Individual Sequence CSV Files

Each split folder also contains one CSV file per sequence:

```text
train/seq_0000.csv
train/seq_0001.csv
...
```

To read one sequence:

```python
seq_df = pd.read_csv("datasets/single_stable_1000x128_error_lv1/train/seq_0000.csv")
```

## How To Iterate Over Sequences

Rows with the same `sequence_id` belong to one independent pulse sequence.

Use `groupby("sequence_id")` when reading from a combined CSV:

```python
for sequence_id, seq_df in train_df.groupby("sequence_id"):
    seq_df = seq_df.sort_values("pulse_index")

    toa = seq_df["TOA"].to_numpy()
    pri = seq_df["PRI"].to_numpy()
    freq = seq_df["Frequency"].to_numpy()
    pw = seq_df["PW"].to_numpy()
    amp = seq_df["Amplitude"].to_numpy()
    aoa = seq_df["AOA"].to_numpy()

    # Use this sequence in your model or algorithm
```

Do not concatenate different `sequence_id` values as one continuous radar signal.

Each `sequence_id` is an independent radar pulse train.

## Important Note About Error Datasets

For clean datasets, each sequence usually has exactly the expected number of pulses.

For error datasets, the observed number of rows in a sequence may be different because:
- missing pulses remove genuine radar pulses
- false/noise pulses add extra observations
- multipath can add echo pulses

So `1000x128` means:

```text
1000 sequences, 128 expected transmitted pulses per sequence
```

It does not always mean every saved sequence has exactly 128 observed rows.

## Recommended Usage

For model development:

```python
train_df = pd.read_csv(f"{dataset_path}/train/sequences.csv")
val_df = pd.read_csv(f"{dataset_path}/val/sequences.csv")
test_df = pd.read_csv(f"{dataset_path}/test/sequences.csv")
```

Use:
- `train` for fitting or learning
- `val` for tuning parameters
- `test` only for final evaluation

For classical algorithms, you can use the same splits for fair comparison.
"""

DATASET_ROOT.mkdir(parents=True, exist_ok=True)

with open(DATASET_ROOT / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)

DATASET_ROOT / "README.md"

WindowsPath('C:/Users/ciftc/OneDrive/Desktop/homeworks/4. year/1/EE 493/repo/Bitirme/datasets/README.md')